# Build the locked Phase 2 retrieval artifact (Colab)
Build the reviewed, deduplicated NewsQA corpus with recursive chunking `512/64` and its BGE-M3 learned-sparse index exactly once. The validated artifact is saved to Google Drive and can be restored in later Phase 2 sessions without re-encoding the 19,263 document chunks.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='62d7e200ca685963e64931863eb2e9a4b047eb16'
HF_REPO_ID='MatchaMacchiato/newsqa_200_11064_v2.0.0'
HF_REVISION='b81c8db6847a23272665946c0c43c72e9a212fd9'  # the v2.0.0 commit; swap for 'v2.0.0' once that tag exists
CHUNK_SIZE=512
CHUNK_OVERLAP=64
CHUNK_STRATEGY='recursive'
SPARSE_ID='bge_m3_sparse'
SPARSE_MODEL='BAAI/bge-m3'
# These four lock the artifact to a byte-exact rebuild, and they belong to
# ONE dataset revision. Restoring the article text changes the chunk count,
# the chunk file, and - because relevant_chunk_ids are re-derived from the
# new chunks - the resolved testset as well. They cannot be known before the
# first build on a new revision, so None means 'print what you observe'; the
# validation cell prints the values to paste back here, which re-locks every
# later rebuild. For v1.0.0 (the truncated corpus) they were:
#   EXPECTED_CHUNKS=19263
#   EXPECTED_CHUNKS_SHA256='9f9c3fd6b645b72c8e2e18764c461d77cccea334cd781dd8eb28950918c474a4'
#   EXPECTED_RESOLVED_SHA256='25471ffb1892210f669e59757537549efd509286f8cf046aafbe403020684b54'
EXPECTED_CHUNKS=None
EXPECTED_CHUNKS_SHA256=None
EXPECTED_RESOLVED_QUESTIONS=1152  # restoration does not touch questions, so this still holds
EXPECTED_RESOLVED_SHA256=None
ARTIFACT_VERSION='phase2-bge-m3-512-64-v2-restored'
FORCE_REBUILD=False
CONTENT=Path('/content')
PROJECT_ROOT=CONTENT/'Text-Mining---NewsQA-RAG'
WORK_ROOT=CONTENT/'newsqa_phase2_index_build'
DATA_ROOT=WORK_ROOT/'data'
VARIANT_ROOT=DATA_ROOT/f'chunk_{CHUNK_SIZE}_{CHUNK_OVERLAP}_{CHUNK_STRATEGY}'
INDEX_ROOT=WORK_ROOT/'index'
CONFIG_ROOT=WORK_ROOT/'configs'
LOG_ROOT=WORK_ROOT/'logs'
BUNDLE_MANIFEST=WORK_ROOT/'phase2_index_bundle_manifest.json'
DRIVE_ROOT=CONTENT/'drive/MyDrive/newsqa_phase2_artifacts'
DRIVE_BUNDLE=DRIVE_ROOT/f'{ARTIFACT_VERSION}.zip'


## 1. Runtime setup
Select a T4 GPU in **Runtime > Change runtime type**, enable notebook access to the private Colab secret `HF_TOKEN`, and ensure Google Drive has enough free space. The Hugging Face token needs read access to the private evaluation dataset. No Gemini key is required for this build.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
# The dataset is public, so a token is optional. Add HF_TOKEN to Colab
# Secrets only to lift anonymous download rate limits.
token=''
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if token:
    os.environ['HF_TOKEN']=token
else:
    print('No HF_TOKEN secret; downloading the public dataset anonymously.')
os.environ['HF_HOME']=str(CONTENT/'hf_cache')
os.environ.update({'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0'})
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
if FORCE_REBUILD and WORK_ROOT.exists(): shutil.rmtree(WORK_ROOT)
for path in [DATA_ROOT,INDEX_ROOT,CONFIG_ROOT,LOG_ROOT]: path.mkdir(parents=True,exist_ok=True)
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
import torch, yaml
assert torch.cuda.is_available(), 'Enable a T4 GPU before building the BGE-M3 index'
print('GPU:',torch.cuda.get_device_name(0),round(torch.cuda.get_device_properties(0).total_memory/2**30,1),'GiB')
print('Pinned repository commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_ROOT,text=True).strip())
print('Drive artifact:',DRIVE_BUNDLE)


In [ ]:
def sha256_file(path,block_size=1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def jsonl_count(path):
    with Path(path).open(encoding='utf-8') as handle: return sum(1 for line in handle if line.strip())
def disk_status():
    usage=shutil.disk_usage(CONTENT); value={'free_gib':round(usage.free/2**30,2),'used_gib':round(usage.used/2**30,2),'total_gib':round(usage.total/2**30,2)}
    print('Disk:',value,flush=True); return value
def run_command(command,label):
    command=[str(value) for value in command]; log_path=LOG_ROOT/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('$',' '.join(command),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
        returncode=process.wait()
    if returncode: raise subprocess.CalledProcessError(returncode,command)
    return log_path
def artifact_paths():
    return {
        'chunks':VARIANT_ROOT/'final_deduplicated/chunks.jsonl',
        'testset_resolved':VARIANT_ROOT/'final_deduplicated/testset_resolved.jsonl',
        'deduplicated_variant_manifest':VARIANT_ROOT/'manifests/deduplicated.variant.json',
        'sparse_index':INDEX_ROOT/f'{SPARSE_ID}.pkl',
        'sparse_config':INDEX_ROOT/f'config_sparse_{SPARSE_ID}.yaml',
        'sparse_variant_manifest':INDEX_ROOT/f'variant_sparse_{SPARSE_ID}.json',
        'index_manifest':INDEX_ROOT/'index_manifest.json',
    }
disk_status()


## 2. Restore an existing Drive artifact
If the versioned bundle already exists, extract it into the fixed Colab work root. Subsequent cells validate every checksum and skip both materialization and document indexing. Set `FORCE_REBUILD=True` only when intentionally replacing a failed local work directory; publish a new artifact version for any semantic configuration change.

In [ ]:
RESTORED_FROM_DRIVE=False
if DRIVE_BUNDLE.exists() and not FORCE_REBUILD:
    print('Restoring existing artifact from Drive:',DRIVE_BUNDLE,flush=True)
    shutil.unpack_archive(DRIVE_BUNDLE,WORK_ROOT)
    RESTORED_FROM_DRIVE=True
else:
    print('No reusable Drive artifact found; a one-time build will run.',flush=True)
print('Restored:',RESTORED_FROM_DRIVE); disk_status()


## 3. Materialize the locked reviewed dataset
Download the immutable private `v1.0.0` source and create only recursive `512/64` chunks plus the reviewed semantic-deduplication outputs. Baseline Chroma is skipped; BM25 is built only because the existing deduplication pipeline uses it while reconstructing the approved dataset.

In [ ]:
base_config=yaml.safe_load((PROJECT_ROOT/'configs/config.yaml').read_text())
base_config.setdefault('chunking',{}).update({'strategy':CHUNK_STRATEGY,'chunk_size':CHUNK_SIZE,'chunk_overlap':CHUNK_OVERLAP})
base_config.setdefault('llm',{}).update({'model':'gemini-3.1-flash-lite','temperature':0.0,'max_tokens':512})
locked_config=CONFIG_ROOT/'phase2_locked_512_64.yaml'
locked_config.write_text(yaml.safe_dump(base_config,sort_keys=False),encoding='utf-8')
paths=artifact_paths()
if not paths['chunks'].exists():
    run_command([sys.executable,'-u','scripts/materialize_evaluation_dataset.py','--repo-id',HF_REPO_ID,'--revision',HF_REVISION,'--config',locked_config,'--output-root',VARIANT_ROOT,'--db-path',WORK_ROOT/'temporary_chroma','--skip-vector-index'],'materialize_locked_dataset')
else:
    print('Locked chunks already exist; skipping materialization.')
def lock(label, observed, expected):
    if expected is None:
        print(f'  RECORD  {label} = {observed!r}')
        return observed
    assert observed == expected, f'{label} drifted: {observed!r} != {expected!r}'
    print(f'  locked  {label} ok')
    return observed
OBSERVED_CHUNKS = lock('EXPECTED_CHUNKS', jsonl_count(paths['chunks']), EXPECTED_CHUNKS)
OBSERVED_CHUNKS_SHA256 = lock('EXPECTED_CHUNKS_SHA256', sha256_file(paths['chunks']), EXPECTED_CHUNKS_SHA256)
lock('EXPECTED_RESOLVED_QUESTIONS', jsonl_count(paths['testset_resolved']), EXPECTED_RESOLVED_QUESTIONS)
OBSERVED_RESOLVED_SHA256 = lock('EXPECTED_RESOLVED_SHA256', sha256_file(paths['testset_resolved']), EXPECTED_RESOLVED_SHA256)
if EXPECTED_CHUNKS is None:
    print()
    print('Paste these into the configuration cell to lock this revision:')
    print('EXPECTED_CHUNKS=' + str(OBSERVED_CHUNKS))
    print("EXPECTED_CHUNKS_SHA256=" + repr(OBSERVED_CHUNKS_SHA256))
    print("EXPECTED_RESOLVED_SHA256=" + repr(OBSERVED_RESOLVED_SHA256))
disk_status()

## 4. Build the BGE-M3 learned-sparse index
This is the expensive one-time step. A T4 encodes all document chunks on CUDA and persists only lexical postings; generation and RAGAS are not run. The builder may produce little output while the GPU is encoding, so check Colab's resource panel rather than interrupting it.

In [ ]:
paths=artifact_paths()
if not paths['index_manifest'].exists() or not paths['sparse_index'].exists():
    run_command([sys.executable,'-u','scripts/build_retrieval_models_index.py','--chunks-path',paths['chunks'],'--base-config',locked_config,'--base-variant-manifest',paths['deduplicated_variant_manifest'],'--output-dir',INDEX_ROOT,'--sparse-ids',SPARSE_ID,'--skip-dense','--device','cuda'],'build_bge_m3_sparse')
else:
    print('BGE-M3 index already exists; skipping document encoding.')
disk_status()


## 5. Validate and record provenance
Validate the index schema, document count, corpus fingerprint, profile manifest, and every required file before packaging. The bundle manifest stores logical relative paths so a later Kaggle/Colab restore can rebase generated absolute paths to its own work root.

In [ ]:
sys.path.insert(0,str(PROJECT_ROOT/'backend'))
from newsqa_rag.indexing.learned_sparse_index import LearnedSparseIndex
paths=artifact_paths()
missing=[name for name,path in paths.items() if not path.exists()]
assert not missing, f'Missing required artifacts: {missing}'
index_manifest=json.loads(paths['index_manifest'].read_text())
assert index_manifest['total_chunks']==OBSERVED_CHUNKS
assert index_manifest['chunks_sha256']==OBSERVED_CHUNKS_SHA256
assert set(index_manifest['sparse_indexes'])=={SPARSE_ID}
index_record=index_manifest['sparse_indexes'][SPARSE_ID]
assert index_record['method']=='bge-m3' and index_record['model_name']==SPARSE_MODEL
loaded_index=LearnedSparseIndex.load(str(paths['sparse_index']),device='cuda')
assert loaded_index.size==OBSERVED_CHUNKS
profile=json.loads(paths['sparse_variant_manifest'].read_text())
assert profile['artifacts']['chunks']['sha256']==OBSERVED_CHUNKS_SHA256
assert profile['artifacts']['testset_resolved']['sha256']==OBSERVED_RESOLVED_SHA256
assert profile['artifacts']['bm25']['sha256']==sha256_file(paths['sparse_index'])
records={name:{'path':str(path.relative_to(WORK_ROOT)),'bytes':path.stat().st_size,'sha256':sha256_file(path)} for name,path in paths.items()}
bundle_manifest={'schema_version':1,'artifact_version':ARTIFACT_VERSION,'created_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'source':{'hf_repo_id':HF_REPO_ID,'hf_revision':HF_REVISION,'repo_commit':REPO_COMMIT},'pipeline':{'chunk_size':CHUNK_SIZE,'chunk_overlap':CHUNK_OVERLAP,'chunk_strategy':CHUNK_STRATEGY,'sparse_id':SPARSE_ID,'sparse_model':SPARSE_MODEL,'device_at_build':'cuda'},'statistics':{'chunks':OBSERVED_CHUNKS,'resolved_questions':EXPECTED_RESOLVED_QUESTIONS},'artifacts':records}
BUNDLE_MANIFEST.write_text(json.dumps(bundle_manifest,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print(json.dumps(bundle_manifest,indent=2)); disk_status()


## 6. Package and save to Drive
Package only the files required by the resolved Phase 2 run. The ZIP does not contain Hugging Face or Gemini credentials, model weights, raw NewsQA staging files, experiment predictions, or RAGAS results. Copying uses a temporary Drive filename so an interrupted upload cannot replace a valid artifact.

In [ ]:
paths=artifact_paths()
local_bundle=CONTENT/f'{ARTIFACT_VERSION}.zip'
if not RESTORED_FROM_DRIVE or FORCE_REBUILD:
    temporary=local_bundle.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
        archive.write(BUNDLE_MANIFEST,BUNDLE_MANIFEST.relative_to(WORK_ROOT))
        for path in paths.values(): archive.write(path,path.relative_to(WORK_ROOT))
        archive.write(locked_config,locked_config.relative_to(WORK_ROOT))
    temporary.replace(local_bundle)
    drive_temporary=DRIVE_BUNDLE.with_suffix('.zip.tmp')
    shutil.copy2(local_bundle,drive_temporary); drive_temporary.replace(DRIVE_BUNDLE)
    print('Saved new immutable artifact to Drive:',DRIVE_BUNDLE,flush=True)
else:
    print('Existing Drive artifact validated; no replacement was written.')
print('Bundle size:',round(DRIVE_BUNDLE.stat().st_size/2**20,1),'MiB')
print('Bundle SHA-256:',sha256_file(DRIVE_BUNDLE)); disk_status()


## Output and reuse
The reusable artifact is stored at `MyDrive/newsqa_phase2_artifacts/phase2-bge-m3-512-64-v1.zip`. Keep this version immutable. Publish it with `scripts/publish_phase2_index_artifact.py`; the Kaggle and Colab Phase 2 notebooks download the pinned private Hugging Face tag, validate `phase2_index_bundle_manifest.json`, rebase generated paths, and skip materialization/index construction. A different dataset revision, chunk configuration, or sparse model requires a new artifact version.